In [0]:
import pyspark as ps
import time
from pyspark.sql.functions import col, sum as spark_sum, isnan, when, count, trim
from pyspark.sql import SparkSession
from pyspark.sql.types import NullType

In [0]:
df = spark.table("workspace.bronze.gho_raw")

In [0]:
display(df)

In [0]:
df.columns

In [0]:
df.printSchema() 

In [0]:
df = spark.table("workspace.bronze.gho_raw")

df = df.filter(df.SpatialDimType == "COUNTRY") \
    .select(
        col("SpatialDim").alias("country_code"),
        col("ParentLocationCode").alias("region"),
        col("TimeDim").alias("year"),
        col("IndicatorCode").alias("indicator_code"),
        col("NumericValue").alias("numeric_value"), 
        col("Low").alias("low"),
        col("High").alias("high")
    )

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.gho_clean")

In [0]:
df = spark.table("workspace.silver.gho_clean")

df.groupBy("country_code", "year", "indicator_code") \
  .count() \
  .filter("count > 1") \
  .groupBy("indicator_code") \
  .agg({"count": "max"}) \
  .show(truncate=False)

In [0]:
df.filter(df.indicator_code == "NUTRITION_ANT_HAZ_NE2") \
  .groupBy("country_code", "year", "indicator_code") \
  .count() \
  .filter("count > 1") \
  .orderBy("count", ascending=False) \
  .limit(1) \
  .show()

In [0]:
df.filter(
    (df.indicator_code == "NUTRITION_ANT_HAZ_NE2") &
    (df.country_code == "NPL") &
    (df.year == "2019")
).show(truncate=False)

In [0]:
df_bronze = spark.table("workspace.bronze.gho_raw")

df_bronze.filter(
    (df_bronze.IndicatorCode == "NUTRITION_ANT_HAZ_NE2") &
    (df_bronze.SpatialDim == "NPL") &
    (df_bronze.TimeDim == 2019)
).select("SpatialDim", "TimeDim", "Dim1Type", "Dim1", "NumericValue") \
 .show(truncate=False)

In [0]:
df_bronze = spark.table("workspace.bronze.gho_raw")
df_bronze.filter(df_bronze.IndicatorCode == "NUTRITION_ANT_HAZ_NE2") \
  .select("Dim1Type", "Dim1") \
  .distinct() \
  .orderBy("Dim1Type", "Dim1") \
  .show(50, truncate=False)

In [0]:
df_bronze.filter(
    (df_bronze.IndicatorCode == "NCDMORT3070") &
    (df_bronze.Dim1.isNull())
).select("SpatialDim", "TimeDim", "Dim1Type", "Dim1", "NumericValue") \
 .limit(5) \
 .show(truncate=False)

In [0]:
df_bronze.filter(df_bronze.IndicatorCode == "NCDMORT3070") \
  .select("Dim1Type", "Dim1") \
  .distinct() \
  .orderBy("Dim1Type", "Dim1") \
  .show(truncate=False)

In [0]:
df_bronze.filter(df_bronze.SpatialDimType == "COUNTRY") \
  .select("IndicatorCode", "Dim1Type", "Dim1") \
  .distinct() \
  .orderBy("IndicatorCode", "Dim1Type", "Dim1") \
  .show(200, truncate=False)

In [0]:
df_bronze = spark.table("workspace.bronze.gho_raw")

filtro = (
    ((df_bronze.IndicatorCode == "GHED_GGHE-D_pc_US_SHA2011") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "HIV_0000000026") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "MDG_0000000029") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "NUTRITION_ANT_HAZ_NE2") & (df_bronze.Dim1.isNull())) |
    ((df_bronze.IndicatorCode == "BP_04") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "M_Est_smk_curr_std") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "NCDMORT3070") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "NCD_BMI_30C") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "NCD_GLUC_04") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "WHOSIS_000001") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "WHOSIS_000002") & (df_bronze.Dim1 == "SEX_BTSX")) |
    ((df_bronze.IndicatorCode == "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV") & (df_bronze.Dim1 == "SEX_FMLE")) |
    ((df_bronze.IndicatorCode == "SDGPM25") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL")) |
    ((df_bronze.IndicatorCode == "WSH_WATER_SAFELY_MANAGED") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL")) |
    ((df_bronze.IndicatorCode == "WSH_SANITATION_SAFELY_MANAGED") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL"))
)

df_silver = df_bronze \
    .filter(df_bronze.SpatialDimType == "COUNTRY") \
    .filter(filtro) \
    .select(
        col("SpatialDim").alias("country_code"),
        col("ParentLocationCode").alias("region"),
        col("TimeDim").alias("year"),
        col("IndicatorCode").alias("indicator_code"),
        col("NumericValue").alias("numeric_value"),
        col("Low").alias("low"),
        col("High").alias("high")
    )

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.gho_clean")

In [0]:
print(f"workspace.silver.gho_clean — {df_silver.count()} lines")


In [0]:
df_silver.filter(
    (df_bronze.IndicatorCode == "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV") &
    (df_bronze.SpatialDim == "BRA") &
    (df_bronze.TimeDim == 2020)
).show(truncate=False)